# Context lifecycle: WRITE → SELECT → COMPRESS → ISOLATE → INVALIDATE
Run the synthetic rehearsal first. It uses the real LangGraph/runtime with scripted source data and LLM responses; it does not measure model quality. Then optionally run OpenAI against live GitHub repositories. Metrics are measured per review, not illustrative token numbers.

In [ ]:
import os, subprocess, sys, importlib
from pathlib import Path
REPO_URL = "https://github.com/swarnamalya-mohan/context-engineering-long-running-agent.git"
BRANCH = "feature/initial-agent-implementation"
if not Path("src/repo_agent").exists():
    folder = Path("context-engineering-long-running-agent")
    if not folder.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(folder)], check=True)
    os.chdir(folder)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# Editable installs do not always refresh an already-running Colab kernel.
repo_root = Path.cwd().resolve()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
from repo_agent.offline import run_offline
print(f"Setup complete: {repo_root}")


## 1. Repeatable rehearsal — no key required
Run the setup cell above first and wait for `Setup complete` before running this cell.
Alpha and Beta pass the base spec. Mandatory SSO makes Alpha ineligible. Allowing AGPL reopens Gamma. SSO documentation is fetched only after activation.

In [ ]:
from repo_agent.offline import run_offline
state = run_offline(".")
print(state["final_recommendation"])


## 2. Show actual context assembly and on-demand skills

In [ ]:
import pandas as pd
contexts = pd.DataFrame(e for e in state["events"] if e["type"] == "context_assembled")
display(contexts[["repo", "loaded_skills", "jit_files", "selected_memory_count", "raw_tokens", "filtered_tokens", "prompt_tokens"]])
contexts.plot.bar(x="repo", y=["raw_tokens", "filtered_tokens"], title="README tokens before and after filtering")


## 3. Show invalidation, reopening, and tool clearing

In [ ]:
display(pd.DataFrame(e for e in state["events"] if e["type"] in
    {"conclusion_invalidated", "openspec_change_applied", "tool_output_cleared"}))
from repo_agent.metrics import candidate_frame, evidence_frame
display(candidate_frame(state))
display(evidence_frame(state))


## 4. Clear history and recover from a durable checkpoint
The checkpoint retains policy, candidate decisions, evidence, and pending work. This demonstrates runtime state recovery; the default LangGraph checkpointer itself is in memory.

In [ ]:
import json
from repo_agent.offline import fixture_runtime
compaction = next(e for e in state["events"] if e["type"] == "compaction")
print("Transient context:", compaction["before_tokens"], "→", compaction["after_tokens"], "tokens")
recovered = json.loads(Path(compaction["durable_checkpoint"]).read_text())
recovered["working_history"] = []
assert recovered["evidence"] == state["evidence"]
print(fixture_runtime(".").synthesize(recovered))


## 5. Optional live OpenAI + GitHub run
Set `RUN_LIVE = True`. Store `OPENAI_API_KEY` and optionally `GITHUB_TOKEN` in Colab Secrets. A live run incurs API usage. Repository facts and rankings can differ from the rehearsal.

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        try:
            os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
        except Exception:
            pass
    except ImportError:
        pass  # Local execution reads environment variables.
    from repo_agent.config import Settings
    from repo_agent.graph import build_graph
    settings = Settings.from_env()
    graph = build_graph(settings=settings, repo_root=".")
    config = {"configurable": {"thread_id": "live-conference"}}
    for update in graph.stream({}, config=config, stream_mode="updates"):
        print(next(iter(update)))
    live_state = graph.get_state(config).values
    display(candidate_frame(live_state))
    display(pd.DataFrame(live_state["events"]))
    print(live_state["final_recommendation"])


## 6. Verify behavioral invariants

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
